# Atlas-Level Differential Expression Analysis

Pseudobulk DESeq2 workflow for an integrated multi-tissue / multi-disease
single-cell RNA-seq atlas.

**Workflow:**
1. Load the atlas (`AnnData`)
2. **Configure** comparisons and analysis parameters ← edit this cell
3. Pseudobulk aggregation per donor × cell-type × disease subset
4. DESeq2 differential expression per comparison
5. Volcano-plot visualisation
6. Combined results table export (upregulated genes only)

Each *comparison* is defined by a `(cell_type(s), disease(s))` **test** group
against a `(cell_type(s), disease(s))` **reference** group. Cell types and
diseases can be single strings or lists of strings, enabling comparisons
across multiple cell types or disease conditions simultaneously. Labels are
generated automatically from the comparison definition.

## 1. Import Required Libraries

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

import scutils

warnings.filterwarnings("ignore", category=FutureWarning)
sc.settings.verbosity = 1

print(f"scutils  {scutils.__version__}")
print(f"scanpy   {sc.__version__}")

## 2. Load the Single-Cell Atlas

In [ ]:
# ── Edit this path to point at your atlas h5ad file ──────────────────────────
ATLAS_PATH = Path("path/to/atlas.h5ad")

adata = sc.read_h5ad(ATLAS_PATH)
print(adata)

# Quick summary of key categorical columns
for col in ["cell_type", "disease", "tissue"]:
    if col in adata.obs.columns:
        counts = adata.obs[col].value_counts()
        print(f"\n{col} ({len(counts)} categories):\n{counts.to_string()}")

## 3. Define Comparison Parameters

**Edit the cell below** to configure every analysis variable.

Each entry in `COMPARISONS` defines one DE test:
- `test` — `(cell_type(s), disease(s))` for the *numerator* group
- `ref` — `(cell_type(s), disease(s))` for the *denominator* group

Cell types and diseases can be a **single string** or a **list of strings**:
- `("AT1", "COPD")` — one cell type, one disease
- `("AT1", ["COPD", "IPF"])` — one cell type, multiple diseases
- `(["AT1", "AT2"], "Control")` — multiple cell types, one disease

Labels are generated automatically as
`{cell_types}_{diseases}_vs_{cell_types}_{diseases}` (values joined with
`-`, spaces replaced with `_`). Per-comparison results are saved in a
subdirectory named after the label.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║                    CONFIGURATION — edit this cell                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Column names in adata.obs ─────────────────────────────────────────────────
SAMPLE_COL    = "donor_id"    # biological replicate (patient / sample ID)
CELL_TYPE_COL = "cell_type"   # cell-type annotation column
DISEASE_COL   = "disease"     # disease / condition label column

# Layer that holds raw integer counts; set to None to use adata.X directly.
# Common options: "counts", "raw_counts", "spliced".
# Note: adata.X must contain raw counts if no layer is specified.
RAW_COUNTS_LAYER: str | None = "counts"

# ── Comparisons ───────────────────────────────────────────────────────────────
# Each dict defines one DE test with two keys:
#   test : (cell_type(s), disease(s)) for the numerator group
#   ref  : (cell_type(s), disease(s)) for the denominator / reference group
#
# Cell types and diseases can be a single string or a list of strings.
# Labels are auto-generated as {cell_types}_{diseases}_vs_{cell_types}_{diseases}
#
# Examples:
#   Same cell type, different disease:
#     test=("Keratinocyte", "Tumor"),  ref=("Keratinocyte", "Normal")
#   Multiple diseases on the test side:
#     test=("AT1", ["COPD", "IPF"]),   ref=("AT1", "Control")
#   Multiple cell types on the reference side:
#     test=("AT1", "COPD"),            ref=(["AT1", "AT2"], "Control")
COMPARISONS: list[dict] = [
    {
        "test": ("Keratinocyte", "Tumor"),
        "ref":  ("Keratinocyte", "Normal"),
    },
    {
        "test": ("T cell", "Tumor"),
        "ref":  ("T cell", "Normal"),
    },
    # ── Add more comparisons here ──────────────────────────────────────────
]

# ── Optional design covariates ────────────────────────────────────────────────
# List of additional columns in adata.obs to include in the DESeq2 design
# formula (e.g. ["sex", "batch"]).  Leave empty to use only the disease column.
EXTRA_COVARIATES: list[str] = []

# ── Pseudobulk settings ───────────────────────────────────────────────────────
MIN_CELLS_PER_GROUP = 10   # minimum cells per sample×group; fewer → dropped

# ── DESeq2 settings ───────────────────────────────────────────────────────────
ALPHA         = 0.05   # adjusted p-value significance threshold
SHRINK_LFC    = True   # apply empirical-Bayes LFC shrinkage (recommended)
N_CPUS: int | None = None  # None → single-threaded; set >1 for speed

# ── Volcano-plot display thresholds ──────────────────────────────────────────
VOLCANO_LFC_CUTOFF  = 0.5   # |log2FC| threshold shown as dashed line
VOLCANO_PVAL_CUTOFF = 0.05  # padj threshold shown as dashed line
TOP_N_GENES         = 10    # number of top genes to label (up + down each)

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("results/de_atlas")   # set to None to skip saving
SAVE_FIGS  = True   # save volcano plots as PDF files

print("Configuration loaded.")
print(f"  Comparisons : {len(COMPARISONS)}")
print(f"  Sample col  : {SAMPLE_COL}")
print(f"  Disease col : {DISEASE_COL}")
print(f"  Cell-type col: {CELL_TYPE_COL}")
print(f"  Raw-counts layer: {RAW_COUNTS_LAYER!r}")
if EXTRA_COVARIATES:
    print(f"  Extra covariates: {EXTRA_COVARIATES}")
print(f"  Output dir  : {OUTPUT_DIR}")

## 4. Pseudobulk Aggregation

For each comparison we:
1. Normalise cell-type and disease entries (single string → one-element list)
2. Auto-generate a file-safe label
3. Combine the test and reference cells into one filtered `AnnData`
4. Add a temporary `_de_group` column that tags each cell as `"test"` or `"ref"`
5. Run `scutils.pp.pseudobulk()` to sum raw counts per `donor × _de_group`

This produces one pseudobulk observation per donor per side of the
comparison, which is the correct input for DESeq2.

In [ ]:
# ── Helpers ──────────────────────────────────────────────────────────────────


def _normalize_group(
    group: tuple,
) -> tuple[list[str], list[str]]:
    """Ensure cell-type and disease entries are always lists.

    Args:
        group: A ``(cell_type, disease)`` tuple where each element can be
            a single string or a list of strings.

    Returns:
        A tuple of two lists: ``(cell_types, diseases)``.
    """
    ct, dis = group
    cts = [ct] if isinstance(ct, str) else list(ct)
    diss = [dis] if isinstance(dis, str) else list(dis)
    return cts, diss


def _make_label(comparison: dict) -> str:
    """Auto-generate a file-safe comparison label.

    Format: ``{cell_types}_{diseases}_vs_{cell_types}_{diseases}``
    where multiple values are joined with ``-`` and spaces are replaced
    with ``_``.
    """
    test_cts, test_diss = _normalize_group(comparison["test"])
    ref_cts, ref_diss = _normalize_group(comparison["ref"])

    def _join(items: list[str]) -> str:
        return "-".join(x.replace(" ", "_") for x in items)

    return f"{_join(test_cts)}_{_join(test_diss)}_vs_{_join(ref_cts)}_{_join(ref_diss)}"


# ── Generate labels for all comparisons ──────────────────────────────────────
labeled_comparisons: dict[str, dict] = {
    _make_label(c): c for c in COMPARISONS
}

print("Comparisons:")
for lbl in labeled_comparisons:
    print(f"  • {lbl}")
print()


def _build_pseudobulk(
    adata: sc.AnnData,
    comparison: dict,
    label: str,
    *,
    sample_col: str,
    cell_type_col: str,
    disease_col: str,
    raw_counts_layer: str | None,
    min_cells: int,
) -> sc.AnnData | None:
    """Subset adata to the two groups, tag them, and pseudobulk.

    Cell types and diseases may be single strings or lists of strings.
    Returns ``None`` and prints a warning if either group has fewer than 2
    unique donors after the *min_cells* filter.
    """
    test_cts, test_diss = _normalize_group(comparison["test"])
    ref_cts, ref_diss = _normalize_group(comparison["ref"])

    # ── Select cells for each side ────────────────────────────────────────
    mask_test = (
        adata.obs[cell_type_col].isin(test_cts)
        & adata.obs[disease_col].isin(test_diss)
    )
    mask_ref = (
        adata.obs[cell_type_col].isin(ref_cts)
        & adata.obs[disease_col].isin(ref_diss)
    )

    n_test = int(mask_test.sum())
    n_ref = int(mask_ref.sum())
    print(
        f"[{label}]  test: {test_cts}/{test_diss} → {n_test:,} cells | "
        f"ref: {ref_cts}/{ref_diss} → {n_ref:,} cells"
    )

    if n_test == 0 or n_ref == 0:
        empty_groups = []
        if n_test == 0:
            empty_groups.append(
                f"test ({', '.join(test_cts)} / {', '.join(test_diss)})"
            )
        if n_ref == 0:
            empty_groups.append(
                f"ref ({', '.join(ref_cts)} / {', '.join(ref_diss)})"
            )
        print(
            f"  ⚠  Skipping '{label}': no cells found for "
            f"{' and '.join(empty_groups)}."
        )
        return None

    # ── Combine and tag ───────────────────────────────────────────────────
    subset = adata[mask_test | mask_ref].copy()
    subset.obs["_de_group"] = "ref"
    subset.obs.loc[mask_test[mask_test | mask_ref], "_de_group"] = "test"

    # If raw counts are in a layer, move them to .X for pseudobulk
    if raw_counts_layer is not None:
        if raw_counts_layer not in subset.layers:
            raise KeyError(
                f"Layer '{raw_counts_layer}' not found. "
                f"Available layers: {list(subset.layers.keys())}"
            )
        subset.X = subset.layers[raw_counts_layer]

    # ── Pseudobulk ────────────────────────────────────────────────────────
    pb = scutils.pp.pseudobulk(
        subset,
        sample_col=sample_col,
        groups_col="_de_group",
        min_cells=min_cells,
        skip_count_check=False,
    )

    # Check we have at least 2 replicates per group
    group_counts = pb.obs["_de_group"].value_counts()
    if group_counts.get("test", 0) < 2 or group_counts.get("ref", 0) < 2:
        print(
            f"  ⚠  Skipping '{label}': fewer than 2 pseudobulk replicates "
            f"per group after min_cells={min_cells} filter.\n"
            f"     Group sizes: {group_counts.to_dict()}"
        )
        return None

    print(
        f"  Pseudobulk: {pb.n_obs} observations "
        f"(test={group_counts.get('test', 0)}, ref={group_counts.get('ref', 0)})"
    )
    return pb


# ── Run pseudobulk for every comparison ──────────────────────────────────────
pseudobulks: dict[str, sc.AnnData] = {}

for label, comp in labeled_comparisons.items():
    pb = _build_pseudobulk(
        adata,
        comp,
        label,
        sample_col=SAMPLE_COL,
        cell_type_col=CELL_TYPE_COL,
        disease_col=DISEASE_COL,
        raw_counts_layer=RAW_COUNTS_LAYER,
        min_cells=MIN_CELLS_PER_GROUP,
    )
    if pb is not None:
        pseudobulks[label] = pb

print(f"\n✓ {len(pseudobulks)}/{len(labeled_comparisons)} comparisons passed pseudobulk QC.")

## 5. Run Differential Expression Analysis

`scutils.tl.deseq2()` fits a DESeq2 Wald-test model on each pseudobulk
object.

The design formula is `~_de_group` plus any `EXTRA_COVARIATES` you
configured.  LFC shrinkage is applied by default (`SHRINK_LFC = True`),
which is strongly recommended for ranking and visualisation.

In [ ]:
# Suppress numerical RuntimeWarnings from pyDESeq2 (overflow, divide-by-zero)
warnings.filterwarnings("ignore", category=RuntimeWarning)

de_results: dict[str, pd.DataFrame] = {}

for label, pb in pseudobulks.items():
    print(f"Running DESeq2 for '{label}' …")

    # Build the design formula
    if EXTRA_COVARIATES:
        # Validate that covariate columns survived pseudobulk propagation
        missing = [c for c in EXTRA_COVARIATES if c not in pb.obs.columns]
        if missing:
            print(
                f"  ⚠  Covariate(s) {missing} not found in pseudobulk obs "
                f"(may have been dropped due to within-group variation). "
                f"Falling back to disease-only design for '{label}'."
            )
            design = "~_de_group"
        else:
            covariate_str = " + ".join(EXTRA_COVARIATES)
            design = f"~{covariate_str} + _de_group"
    else:
        design = "~_de_group"

    print(f"  Design: {design}")

    try:
        results = scutils.tl.deseq2(
            pb,
            design=design,
            contrast=["_de_group", "test", "ref"],
            alpha=ALPHA,
            shrink_lfc=SHRINK_LFC,
            ref_level=["_de_group", "ref"],
            n_cpus=N_CPUS,
            quiet=True,
        )
    except Exception as exc:
        print(f"  ✗ DESeq2 failed for '{label}': {exc}")
        continue

    n_sig = int((results["padj"] < ALPHA).sum())
    n_up  = int(((results["padj"] < ALPHA) & (results["log2FoldChange"] > 0)).sum())
    n_dn  = int(((results["padj"] < ALPHA) & (results["log2FoldChange"] < 0)).sum())
    print(
        f"  ✓ Significant genes: {n_sig}  "
        f"(up: {n_up}, down: {n_dn})"
    )

    de_results[label] = results

print(f"\n✓ DE completed for {len(de_results)}/{len(pseudobulks)} comparisons.")

## 6. Visualise Results: Volcano Plots

One volcano plot is produced per comparison.  Genes are coloured by
significance and fold-change direction; the top `TOP_N_GENES` up- and
down-regulated hits are annotated by name.

Plots are optionally saved to `OUTPUT_DIR` as PDF files.

In [ ]:
for label, results in de_results.items():
    # Look up the comparison metadata for the plot title
    comp_meta = labeled_comparisons[label]
    test_cts, test_diss = _normalize_group(comp_meta["test"])
    ref_cts, ref_diss = _normalize_group(comp_meta["ref"])
    title = (
        f"{', '.join(test_cts)} / {', '.join(test_diss)}  vs  "
        f"{', '.join(ref_cts)} / {', '.join(ref_diss)}\n"
        f"(n_sig padj<{ALPHA}: "
        f"{int((results['padj'] < ALPHA).sum())} genes)"
    )

    fig = scutils.pl.volcano_plot(
        results,
        pval_col="padj",
        lfc_col="log2FoldChange",
        pval_cutoff=VOLCANO_PVAL_CUTOFF,
        lfc_cutoff=VOLCANO_LFC_CUTOFF,
        top_n_up=TOP_N_GENES,
        top_n_down=TOP_N_GENES,
        figsize=(10, 7),
    )
    fig.axes[0].set_title(title, fontsize=11, pad=10)
    fig.tight_layout()

    if OUTPUT_DIR is not None and SAVE_FIGS:
        comp_dir = OUTPUT_DIR / label
        comp_dir.mkdir(parents=True, exist_ok=True)
        out_path = comp_dir / f"volcano_{label}.pdf"
        fig.savefig(out_path, bbox_inches="tight")
        print(f"Saved: {out_path}")

    plt.show()
    plt.close(fig)

## 7. Export Results Table

Each comparison's full results (all genes) are saved to a **subdirectory**
named after its auto-generated label:
`<OUTPUT_DIR>/<label>/de_<label>.csv`

A **combined table** containing only significantly **upregulated** genes
across all comparisons is saved to `<OUTPUT_DIR>/de_all_comparisons_upregulated.csv`.

Columns per row:
`gene`, `baseMean`, `log2FoldChange`, `lfcSE`, `stat`, `pvalue`, `padj`,
`comparison`, `test_cell_type`, `test_disease`, `ref_cell_type`, `ref_disease`

In [ ]:
# ── Build a long-form combined table ─────────────────────────────────────────
frames: list[pd.DataFrame] = []

for label, results in de_results.items():
    comp_meta = labeled_comparisons[label]
    test_cts, test_diss = _normalize_group(comp_meta["test"])
    ref_cts, ref_diss = _normalize_group(comp_meta["ref"])

    df = results.copy()
    df.index.name = "gene"
    df = df.reset_index()

    df["comparison"]     = label
    df["test_cell_type"] = ", ".join(test_cts)
    df["test_disease"]   = ", ".join(test_diss)
    df["ref_cell_type"]  = ", ".join(ref_cts)
    df["ref_disease"]    = ", ".join(ref_diss)

    frames.append(df)

combined = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

# ── Save ──────────────────────────────────────────────────────────────────────
if OUTPUT_DIR is not None and not combined.empty:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # One file per comparison (full results in a comparison-specific subdirectory)
    for label, grp in combined.groupby("comparison"):
        comp_dir = OUTPUT_DIR / label
        comp_dir.mkdir(parents=True, exist_ok=True)
        out_path = comp_dir / f"de_{label}.csv"
        grp.to_csv(out_path, index=False)
        print(f"Saved: {out_path}")

    # Combined file — significantly upregulated genes only
    upreg = combined[
        (combined["padj"] < ALPHA)
        & (combined["log2FoldChange"] > VOLCANO_LFC_CUTOFF)
    ].sort_values(["comparison", "padj"])

    if not upreg.empty:
        combined_path = OUTPUT_DIR / "de_all_comparisons_upregulated.csv"
        upreg.to_csv(combined_path, index=False)
        print(f"\nSaved combined upregulated table ({len(upreg)} genes): {combined_path}")
    else:
        print("\nNo significantly upregulated genes to save in the combined table.")

# ── Display top upregulated hits inline ──────────────────────────────────────
if not combined.empty:
    upreg_display = (
        combined[
            (combined["padj"] < ALPHA)
            & (combined["log2FoldChange"] > VOLCANO_LFC_CUTOFF)
        ]
        .sort_values("padj")
        .groupby("comparison")
        .head(10)
    )
    if not upreg_display.empty:
        print(
            f"\nTop 10 upregulated genes per comparison "
            f"({len(upreg_display)} rows shown):"
        )
        display(
            upreg_display[
                ["comparison", "gene", "log2FoldChange", "padj",
                 "test_cell_type", "test_disease", "ref_cell_type", "ref_disease"]
            ]
            .style
            .format({"log2FoldChange": "{:.3f}", "padj": "{:.2e}"})
            .background_gradient(subset="log2FoldChange", cmap="Reds")
        )
    else:
        print("No significantly upregulated genes found.")
else:
    print("No results to display.")

---

## Summary

| Step | Function | Key parameters |
|------|----------|----------------|
| Load atlas | `sc.read_h5ad()` | `ATLAS_PATH` |
| Pseudobulk | `scutils.pp.pseudobulk()` | `SAMPLE_COL`, `MIN_CELLS_PER_GROUP` |
| DE analysis | `scutils.tl.deseq2()` | `ALPHA`, `SHRINK_LFC`, `EXTRA_COVARIATES` |
| Volcano plot | `scutils.pl.volcano_plot()` | `VOLCANO_LFC_CUTOFF`, `VOLCANO_PVAL_CUTOFF`, `TOP_N_GENES` |
| Export (per comparison) | `pd.DataFrame.to_csv()` | `OUTPUT_DIR/<label>/` |
| Export (combined upreg.) | `pd.DataFrame.to_csv()` | `OUTPUT_DIR/de_all_comparisons_upregulated.csv` |

**Adding a new comparison:** add a dict to `COMPARISONS` in the
configuration cell and re-run from section 4 onwards. Labels are generated
automatically.

**Multi-value comparisons:** cell types and diseases accept lists, e.g.
`test=("AT1", ["COPD", "IPF"])` or `ref=(["AT1", "AT2"], "Control")`.